In [1]:
import sys
sys.path.insert(0, "..")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from src.data.fetch import fetch_stock_data
from src.indicators.pipeline import compute_all_indicators

C:\Users\devgi\AppData\Roaming\Python\Python313\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [2]:
df = fetch_stock_data("RELIANCE.NS", period="1y")
df = compute_all_indicators(df)
last_6m = df.iloc[-126:]   # last ~6 months for clarity

Fetching RELIANCE.NS  (1y)...
Downloaded 250 trading days
Range : 2025-07-03 -> 2026-07-03
Latest Close : ₹1,304.00
Saved to C:\Users\devgi\InvestIQ\notebooks\..\data\raw\RELIANCE_NS_1y.csv


In [3]:

fig = plt.figure(figsize=(16, 14))
gs = gridspec.GridSpec(4, 1, height_ratios=[3, 1, 1, 1], hspace=0.08)

<Figure size 1600x1400 with 0 Axes>

# ── Panel 1: Price + Bollinger Bands + MAs ────────────────────────

In [4]:
ax1 = fig.add_subplot(gs[0])
ax1.plot(last_6m.index, last_6m["Close"],    color="#1565C0", lw=1.5, label="Close", zorder=3)
ax1.plot(last_6m.index, last_6m["SMA_20"],   color="#FF6F00", lw=1,   label="SMA 20", linestyle="--")
ax1.plot(last_6m.index, last_6m["SMA_50"],   color="#E53935", lw=1,   label="SMA 50", linestyle="--")
ax1.plot(last_6m.index, last_6m["BB_Upper"], color="#78909C", lw=0.8, linestyle=":")
ax1.plot(last_6m.index, last_6m["BB_Lower"], color="#78909C", lw=0.8, linestyle=":")
ax1.fill_between(last_6m.index, last_6m["BB_Upper"], last_6m["BB_Lower"],
                  alpha=0.05, color="#78909C")
ax1.set_title("Reliance — Price, Moving Averages, Bollinger Bands", fontsize=12)
ax1.set_ylabel("Price (₹)")
ax1.legend(loc="upper left", fontsize=9)
ax1.grid(alpha=0.2)
ax1.set_xticklabels([])

[Text(20454.0, 0, ''),
 Text(20485.0, 0, ''),
 Text(20513.0, 0, ''),
 Text(20544.0, 0, ''),
 Text(20574.0, 0, ''),
 Text(20605.0, 0, ''),
 Text(20635.0, 0, '')]

# ── Panel 2: RSI ──────────────────────────────────────────────────

In [5]:
ax2 = fig.add_subplot(gs[1])
ax2.plot(last_6m.index, last_6m["RSI"], color="#7B1FA2", lw=1.2)
ax2.axhline(70, color="#E53935", linestyle="--", lw=0.8, alpha=0.7)
ax2.axhline(30, color="#2E7D32", linestyle="--", lw=0.8, alpha=0.7)
ax2.axhline(50, color="gray",    linestyle="--", lw=0.5, alpha=0.5)
ax2.fill_between(last_6m.index, last_6m["RSI"], 70,
                  where=(last_6m["RSI"] >= 70), color="#E53935", alpha=0.2)
ax2.fill_between(last_6m.index, last_6m["RSI"], 30,
                  where=(last_6m["RSI"] <= 30), color="#2E7D32", alpha=0.2)
ax2.set_ylabel("RSI (14)")
ax2.set_ylim(0, 100)
ax2.grid(alpha=0.2)
ax2.set_xticklabels([])


[Text(20454.0, 0, ''),
 Text(20485.0, 0, ''),
 Text(20513.0, 0, ''),
 Text(20544.0, 0, ''),
 Text(20574.0, 0, ''),
 Text(20605.0, 0, ''),
 Text(20635.0, 0, '')]

# ── Panel 3: MACD ─────────────────────────────────────────────────

In [6]:
ax3 = fig.add_subplot(gs[2])
ax3.plot(last_6m.index, last_6m["MACD"],        color="#1565C0", lw=1.2, label="MACD")
ax3.plot(last_6m.index, last_6m["MACD_Signal"], color="#FF6F00", lw=1.2, label="Signal")
ax3.bar(last_6m.index, last_6m["MACD_Hist"],
        color=["#43A047" if v >= 0 else "#E53935" for v in last_6m["MACD_Hist"]],
        alpha=0.6, width=1)
ax3.axhline(0, color="gray", lw=0.7)
ax3.set_ylabel("MACD")
ax3.legend(loc="upper left", fontsize=9)
ax3.grid(alpha=0.2)
ax3.set_xticklabels([])

[Text(20454.0, 0, ''),
 Text(20485.0, 0, ''),
 Text(20513.0, 0, ''),
 Text(20544.0, 0, ''),
 Text(20574.0, 0, ''),
 Text(20605.0, 0, ''),
 Text(20635.0, 0, '')]

# ── Panel 4: Volume Ratio ─────────────────────────────────────────

In [7]:
ax4 = fig.add_subplot(gs[3])
colours = ["#43A047" if v >= 1 else "#78909C" for v in last_6m["Volume_Ratio"]]
ax4.bar(last_6m.index, last_6m["Volume_Ratio"], color=colours, alpha=0.8, width=1)
ax4.axhline(1, color="gray", lw=0.8, linestyle="--")
ax4.set_ylabel("Vol Ratio")
ax4.grid(alpha=0.2)

fig.savefig("../data/charts/indicators_chart.png", dpi=150, bbox_inches="tight")
plt.show()